# backwardCGM-PD — fMRI trên Kaggle

Notebook độc lập để dựng lại network Subject 14/15 từ selected model đã lưu, hoặc refit nếu bạn cung cấp hai time-series matrix 36 cột. Hãy **Add Input** dataset `backwardCGM-PD`.

In [ ]:
import importlib.util, subprocess, sys
required = {"rdata": "rdata>=0.11", "networkx": "networkx>=3.0", "joblib": "joblib>=1.3"}
missing = [pkg for module, pkg in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print("Dependencies: OK")

In [ ]:
from pathlib import Path
import json, shutil, zipfile
from IPython.display import Image, display

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working/backwardCGM-PD-3")
RESULTS = Path("/kaggle/working/fmri-results")
RESULTS.mkdir(parents=True, exist_ok=True)

# Khôi phục checkpoint từ output của một Kaggle Version trước nếu đã Add Input.
checkpoint_archives = list(INPUT_ROOT.rglob("fmri-results.zip"))
checkpoint_files = list(INPUT_ROOT.rglob("models.json"))
if checkpoint_archives:
    with zipfile.ZipFile(checkpoint_archives[0]) as archive:
        archive.extractall(RESULTS)
    print("Restored checkpoint:", checkpoint_archives[0])
elif checkpoint_files:
    shutil.copytree(checkpoint_files[0].parent, RESULTS, dirs_exist_ok=True)
    print("Restored checkpoint:", checkpoint_files[0])

archives = list(INPUT_ROOT.rglob("backwardCGM-PD-kaggle-dataset.zip"))
scripts = list(INPUT_ROOT.rglob("python-port/experiments/fmri.py"))
if archives:
    WORK_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archives[0]) as archive:
        archive.extractall(WORK_ROOT)
elif scripts:
    shutil.copytree(scripts[0].parents[2], WORK_ROOT, dirs_exist_ok=True)
else:
    raise FileNotFoundError("Hãy Add Input dataset backwardCGM-PD")

PORT_ROOT = WORK_ROOT / "python-port"
SAVED_MODELS = WORK_ROOT / "applications/fMRIdata/output-fMRI/36variables"
required_models = [SAVED_MODELS / f"res{s}.36.tauc.RData" for s in (14, 15)]
if not (PORT_ROOT / "experiments/fmri.py").exists() or not all(p.exists() for p in required_models):
    raise FileNotFoundError("Dataset fMRI không đầy đủ")
print("Dataset: OK\nSaved models:", SAVED_MODELS, "\nOutput:", RESULTS)

## Dữ liệu refit tùy chọn

Giữ `None` để dựng hình từ model lưu. Nếu có dữ liệu gốc, đặt đường dẫn `.npy` hoặc `.csv`; mỗi file phải có 36 cột theo thứ tự 18 ROI trái rồi 18 ROI phải tương ứng.

In [ ]:
SUBJECT14_DATA = None  # Ví dụ: Path('/kaggle/input/backwardcgm-pd/subject14.npy')
SUBJECT15_DATA = None  # Ví dụ: Path('/kaggle/input/backwardcgm-pd/subject15.npy')

In [ ]:
command = [
    sys.executable, "-u", str(PORT_ROOT / "experiments/fmri.py"),
    "--saved-dir", str(SAVED_MODELS), "--output-dir", str(RESULTS),
    "--alpha", "0.05", "--itmax", "1000", "--resume", "--verbose",
]
if SUBJECT14_DATA is not None:
    command.extend(["--subject14-data", str(SUBJECT14_DATA)])
if SUBJECT15_DATA is not None:
    command.extend(["--subject15-data", str(SUBJECT15_DATA)])
print("Running:", " ".join(command))
subprocess.run(command, cwd=PORT_ROOT, check=True)

In [ ]:
for subject in (14, 15):
    print(f"Subject {subject}")
    for component in ("asymmetric-left", "asymmetric-right", "symmetric-within", "symmetric-between"):
        figure = RESULTS / f"subject-{subject}-{component}.png"
        if figure.exists():
            display(Image(filename=str(figure), width=700))
models = json.loads((RESULTS / "models.json").read_text(encoding="utf-8"))
display(models)
archive = shutil.make_archive("/kaggle/working/fmri-results", "zip", root_dir=RESULTS)
print("Download:", archive)

**Checkpoint:** `models.json` được ghi atomically sau mỗi Subject. Nếu refit bị ngắt, Subject đã hoàn tất sẽ được bỏ qua khi resume. Để tiếp tục trong session mới, hãy **Save Version** rồi **Add Input** output cũ; notebook tự khôi phục `fmri-results.zip` hoặc `models.json`.

**Blocker:** dataset hiện chỉ có selected models, không có time series 36 biến. Vì vậy mặc định notebook chỉ dựng network, không chạy lại model selection.